# Smart Resume Ranking System

This notebook builds a simple resume screening model using candidate features and recruiter decisions. It focuses on supervised classification and model evaluation.

## Goals

- Explore the resume screening dataset
- Preprocess data and encode categorical labels
- Train and compare Logistic Regression and Random Forest classifiers
- Evaluate model performance with accuracy, classification report, and confusion matrix

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

# Load dataset

df = pd.read_csv('AI_Resume_Screening.csv')
print('Dataset loaded: rows =', df.shape[0], 'columns =', df.shape[1])

In [ ]:
# Preview the dataset and missing values

display(df.head())
print('\nMissing values per column:')
print(df.isna().sum())

In [ ]:
# Fix missing values and create a combined text column for later use

df['Certifications'].fillna('', inplace=True)
df['Skills'] = df['Skills'].fillna('')
df['Education'] = df['Education'].fillna('')

df['combined_text'] = df['Skills'] + ', ' + df['Education']
df.drop(['Skills', 'Education'], axis=1, inplace=True)

display(df.head())

In [ ]:
# Label encode categorical columns

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

if df['Job Role'].dtype == object:
    df['Job Role'] = le.fit_transform(df['Job Role'])
else:
    df['Job Role'] = df['Job Role']

recruiter_le = LabelEncoder()
df['Recruiter Decision'] = recruiter_le.fit_transform(df['Recruiter Decision'])

print('Encoded recruiter decision classes:', list(recruiter_le.classes_))

In [ ]:
# Visualize target distribution and AI score distribution by decision

plt.figure(figsize=(8, 4))
sns.countplot(x='Recruiter Decision', data=df, palette='coolwarm')
plt.title('Recruiter Decision Counts')
plt.xlabel('Recruiter Decision')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x='Recruiter Decision', y='AI Score (0-100)', data=df, palette='magma')
plt.title('AI Score Distribution by Recruiter Decision')
plt.show()

In [ ]:
# Select numerical features for classification

X = df[["Experience (Years)", "Projects Count", "Salary Expectation ($)"]]
y = df['Recruiter Decision']

print('Feature set shape:', X.shape)
print('Target distribution:')
print(y.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(criterion='entropy', n_estimators=250, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'report': classification_report(y_test, y_pred, target_names=recruiter_le.classes_, zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }

for name, metrics in results.items():
    print('#' * 50)
    print(name)
    print('Accuracy:', round(metrics['accuracy'], 4))
    print('\nClassification report:\n', metrics['report'])

In [ ]:
# Confusion matrix for the best model

best_model = max(results.items(), key=lambda item: item[1]['accuracy'])[0]
cm = results[best_model]['confusion_matrix']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=recruiter_le.classes_, yticklabels=recruiter_le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model}')
plt.show()

print('Best performing model:', best_model)

## Summary

- The dataset was cleaned and encoded correctly.
- Two classification models were trained and compared.
- The best model is shown with its confusion matrix and classification metrics.

**Next improvements:**
- add more meaningful features from resume text
- use NLP to process skills and education
- tune model hyperparameters for better accuracy